In [7]:
import pandas as pd

df = pd.read_csv(r'C:\GIS_temp\dot_speeds_raw.csv')

# Fix date parsing
df['DATA_AS_OF'] = pd.to_datetime(df['DATA_AS_OF'])
df = df[(df['DATA_AS_OF'] >= '2025-02-15') & (df['DATA_AS_OF'] <= '2026-02-15')]

# Clean speeds
df = df[(df['SPEED'] > 0) & (df['SPEED'] < 100)]

# Congestion flag
df['congested'] = (df['SPEED'] < 10).astype(int)

# Aggregate to one row per segment, keep LINK_POINTS and BOROUGH
seg = df.groupby('ID').agg(
    obs=('congested', 'count'),
    cong=('congested', 'sum'),
    link_points=('LINK_POINTS', 'first'),
    borough=('BOROUGH', 'first')
).reset_index()

seg['cong_share'] = seg['cong'] / seg['obs']

print(f"Total rows in df: {len(df)}")
print(f"Total segments: {len(seg)}")
print(seg.head(10))
print(f"Congestion share range: {seg['cong_share'].min():.3f} to {seg['cong_share'].max():.3f}")

C:\Users\Arv\AppData\Local\Temp\ipykernel_27980\3163818674.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['DATA_AS_OF'] = pd.to_datetime(df['DATA_AS_OF'])


Total rows in df: 9914653
Total segments: 112
    ID     obs   cong                                        link_points  \
0  119   88900   5525  40.70631,-74.01501 40.705380,-74.01528 40.7049...   
1  124   83871   3495  40.68036,-74.00441001 40.6822,-74.0057201 40.6...   
2  126  102948    478  40.8271606,-73.84993 40.82771,-73.84671 40.828...   
3  129  103373     93  40.8240706,-73.874311 40.8247,-73.86959 40.825...   
4  137  103437   2469  40.8242005,-73.874361 40.8249804,-73.868411 40...   
5  140   95822  15160  40.79789,-73.91988 40.79771,-73.92004 40.79758...   
6  141  102689   4496  40.772251,-73.919891 40.77391,-73.9222 40.7747...   
7  142  102656    249  40.83037,-73.85062 40.82996,-73.849251 40.8294...   
8  145   92967  32610  40.7081105,-73.99944 40.7084705,-73.99884 40.7...   
9  148   78592   3765  40.69153,-73.99911 40.6922605,-73.99937 40.692...   

     borough  cong_share  
0  Manhattan    0.062148  
1  Manhattan    0.041671  
2      Bronx    0.004643  
3      Br

In [5]:
print(f"Total rows in df: {len(df)}")
print(f"Total segments: {len(seg)}")
print(seg.head(10))
print(f"Congestion share range: {seg['cong_share'].min():.3f} to {seg['cong_share'].max():.3f}")

Total rows in df: 1596501
Total segments: 124
    ID    obs   cong  cong_share
0    1  12928  12928    1.000000
1    2  12928  12928    1.000000
2    3  12928  12928    1.000000
3    4  12928  12928    1.000000
4  106  12928  12928    1.000000
5  119  12928   2452    0.189666
6  124  12928   2819    0.218054
7  126  12928    107    0.008277
8  129  12928     17    0.001315
9  137  12928    421    0.032565
Congestion share range: 0.000 to 1.000


In [8]:
import pandas as pd

df = pd.read_csv(r'C:\GIS_temp\dot_speeds_raw.csv')
print(df.columns.tolist())
print(df.head(3))

['ID', 'SPEED', 'DATA_AS_OF', 'LINK_POINTS', 'BOROUGH', 'LINK_NAME']
    ID  SPEED               DATA_AS_OF  \
0  399   7.45  2026 Feb 14 11:58:13 PM   
1  376  48.46  2026 Feb 14 11:58:13 PM   
2  375  48.46  2026 Feb 14 11:58:13 PM   

                                         LINK_POINTS        BOROUGH  \
0  40.8011005,-73.92846 40.80151,-73.93066 40.801...      Manhattan   
1  40.61052,-74.09769 40.610561,-74.09586 40.6102...  Staten Island   
2  40.61052,-74.09769 40.6101,-74.10893 40.61028,...  Staten Island   

                                       LINK_NAME  
0  TBB W - FDR S MANHATTAN TRUSS - E116TH STREET  
1            SIE E CLOVE ROAD - FINGERBOARD ROAD  
2              SIE E BRADLEY AVENUE - CLOVE ROAD  


In [9]:
import geopandas as gpd
from shapely.geometry import LineString

def parse_link_points(s):
    try:
        coords = [tuple(map(float, p.split(','))) for p in s.strip().split(' ') if ',' in p]
        coords = [(lon, lat) for lat, lon in coords]  # flip to (lon, lat) for shapely
        return LineString(coords) if len(coords) >= 2 else None
    except:
        return None

seg['geometry'] = seg['link_points'].apply(parse_link_points)
seg = seg[seg['geometry'].notna()]
gdf = gpd.GeoDataFrame(seg, geometry='geometry', crs='EPSG:4326')

# Load NTA boundaries
nta = gpd.read_file(r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\2020_Neighborhood_Tabulation_Areas_(NTAs)_20260305.geojson')
print(nta.columns.tolist())
print(f"NTAs loaded: {len(nta)}")

[':id', ':version', ':created_at', ':updated_at', 'borocode', 'boroname', 'countyfips', 'nta2020', 'ntaname', 'ntaabbrev', 'ntatype', 'cdta2020', 'cdtaname', 'shape_leng', 'shape_area', 'geometry']
NTAs loaded: 262


In [11]:
joined = gpd.sjoin(gdf, nta[['nta2020', 'geometry']], how='left', predicate='intersects')

result = joined.groupby('nta2020')['cong_share'].mean().reset_index()
result.columns = ['nta2020', 'traf_congestion_proxy']

print(f"NTAs with coverage: {len(result)}")
print(result.sort_values('traf_congestion_proxy', ascending=False).head(10))

result.to_csv(
    r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\nta_traffic_congestion.csv',
    index=False
)
print("Saved.")

NTAs with coverage: 138
    nta2020  traf_congestion_proxy
58   MN0602               0.896899
56   MN0502               0.896899
98   QN0905               0.896899
55   MN0501               0.896899
107  QN1203               0.568751
52   MN0303               0.547335
51   MN0302               0.547335
99   QN1001               0.501495
79   QN0503               0.472402
81   QN0574               0.472402
Saved.
